<a href="https://colab.research.google.com/github/edaska/Stochastic_Processes_-_Optimization_in_Machine_Learning/blob/main/lab7/Stochastic_Processes_%26_Optimization_in_Machine_Learning_(Lab_7_Markov_Decision_Processes).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><b>Markov Decision Processes</b></h1>

<p align="justify">The problem you will examine is the Frozen Lake problem with a grid size of 8 x 8.</p>


<h2><b>Getting Familiar with the <i>Gymnasium</i> Library</b></h2>

In [5]:
import numpy as np
np.bool8 = np.bool_
!pip install setuptools
!pip install gymnasium
import gymnasium as gym
from gym import wrappers

With the following command, you initialize the grid:



In [6]:
env = gym.make("FrozenLake-v1", map_name="8x8", render_mode="ansi")

With the following commands, you can reset the Agent to its initial position and visualize the grid and the Agent’s position.

In [7]:
obs, info = env.reset()
print(env.render())


SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG



Next, you can define the next action randomly and the Agent takes one step.

In [8]:
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

print("Random action:", action)
print("New state:", obs)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print(env.render())

Random action: 2
New state: 8
Reward: 0
Terminated: False
Truncated: False
  (Right)
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG



<h2><b>Questions</b></h2>

<ul>
<li>After reading <a href="https://towardsdatascience.com/q-learning-for-beginners-2837b777741/">this</a>, briefly describe the Frozen Lake problem as an optimization problem. What is the agent’s goal?</li>
<li>Briefly describe the <i>Policy Iteration</i> and <i>Value Iteration</i> algorithms, emphasizing the different way in which they approach solving the problem. Is it guaranteed that the two algorithms will converge to the optimal policy? If yes, do they lead to the same or to different policies?</li>
<li>Run the program which solves the Frozen Lake problem using the <i>Policy Iteration</i> and <i>Value Iteration</i> algorithms, respectively. Which method converges to the optimal solution in fewer steps? Check the execution time of each program using the <i>time</i> command. What can you conclude regarding the complexity of each algorithm?</li>
</ul>

**After reading [this](https://towardsdatascience.com/q-learning-for-beginners-2837b777741/), briefly describe the Frozen Lake problem as an optimization problem. What is the agent’s goal?**

Based on the article, the Frozen Lake problem is an optimization problem where an agent must learn the best sequence of actions to move from the starting tile S to the goal tile G in a grid environment.

The environment has 16 states because the lake is a 4×4 grid, and in each state the agent can choose one of 4 actions: left, down, right, or up. Some tiles are safe frozen tiles, while others are holes. If the agent falls into a hole, the episode ends unsuccessfully. If it reaches the goal, it receives a reward of 1, otherwise, the reward is 0.

So, the optimization objective is to learn an optimal policy, meaning the best action to take in each state. In this example, this is done by updating a Q-table, where each value estimates how good a specific action is in a specific state.

The agent’s goal is to reach the goal tile G while avoiding holes, ideally using the minimum number of actions. In the article’s example, the optimal solution requires 6 actions, such as:

RIGHT → RIGHT → DOWN → DOWN → DOWN → RIGHT

In conclusion, we can say that, the agent tries to optimize its decisions so that it reaches the goal safely and as efficiently as possible.

**Briefly describe the Policy Iteration and Value Iteration algorithms, emphasizing the different way in which they approach solving the problem. Is it guaranteed that the two algorithms will converge to the optimal policy? If yes, do they lead to the same or to different policies?**

Policy Iteration has two steps. First, it performs **policy evaluation**. It starts with an arbitrary policy $μ_n$. And then calculates the cost-to-go values under the policy $J^μ(i)$. This is called the critic step in actor-critic interpretation, because it evaluates how good or bad the current policy is. Then it performs **policy improvement**. In this step it calculates Q-factors for alternative actions $Q^{μ_n}(i, α)$. The policy is updated by choosing, for each state, the action with the minimum Q-factor. This process repeats until the policy no longer changes. This is the actor step, because it changes the agent's decisions. In simple words, Policy Iteration improves the policy directly step by step.

Value Iteration , instead of repeatedly evaluating and improving a full policy, directly updated the cost-to-go estimation. Starts with arbitrary values $J_0(i)$ for all states and repeatedly updates them using Bellman’s equation until the values converge. After the cost-to-go values become stable, the optimal policy is extracted by choosing the action with the best Q-factor in each state. So, Value Iteration focuses first on estimating the optimal value function, and only then derives the policy.

The main difference is therefore this: Policy Iteration alternates between evaluating and improving a policy, while Value Iteration repeatedly improves the value estimates and derives the policy at the end.

Under the finite-state and finite-action, these algorithms are guaranteed to converge to an optimal policy. Policy Iteration converges to an optimal policy in finite steps because the state and action spaces are finite. Value Iteration also converges theoretically as the number of iterations goes to infinity, or approximately in practice when a small convergence tolerance is reached.

They solve the same optimization problem, so they lead to the same optimal cost or value function. Usually, they also lead to the same optimal policy. However, if there are multiple actions with exactly the same optimal value, then they may return different but equally optimal policies.

In [9]:
import numpy as np
import gymnasium as gym


def run_episode(env, policy, gamma=1.0, max_steps=1000):
    obs, info = env.reset()
    total_reward = 0.0
    step_idx = 0

    while step_idx < max_steps:
        action = int(policy[obs])
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += (gamma ** step_idx) * reward
        step_idx += 1

        if terminated or truncated:
            break

    return total_reward, step_idx


def evaluate_policy(env, policy, gamma=1.0, n=5000):
    rewards = []
    steps = []

    for _ in range(n):
        r, s = run_episode(env, policy, gamma=gamma)
        rewards.append(r)
        steps.append(s)

    return np.mean(rewards), np.mean(steps)


def compute_policy_v(env, policy, gamma=0.99, eps=1e-10):
    base_env = env.unwrapped
    nS = env.observation_space.n
    v = np.zeros(nS, dtype=np.float64)

    while True:
        prev_v = v.copy()

        for s in range(nS):
            a = int(policy[s])
            v[s] = sum(
                p * (r + (0.0 if terminated else gamma * prev_v[s_]))
                for p, s_, r, terminated in base_env.P[s][a]
            )

        if np.max(np.abs(prev_v - v)) < eps:
            break

    return v


def extract_policy(env, v, gamma=0.99):
    base_env = env.unwrapped
    nS = env.observation_space.n
    nA = env.action_space.n
    policy = np.zeros(nS, dtype=int)

    for s in range(nS):
        q_sa = np.zeros(nA, dtype=np.float64)

        for a in range(nA):
            q_sa[a] = sum(
                p * (r + (0.0 if terminated else gamma * v[s_]))
                for p, s_, r, terminated in base_env.P[s][a]
            )

        policy[s] = int(np.argmax(q_sa))

    return policy


def policy_iteration(env, gamma=0.99, max_iterations=1000):
    nS = env.observation_space.n
    nA = env.action_space.n
    policy = np.random.choice(nA, size=nS)

    for i in range(max_iterations):
        old_policy = policy.copy()
        v = compute_policy_v(env, policy, gamma=gamma)
        policy = extract_policy(env, v, gamma=gamma)

        if np.array_equal(policy, old_policy):
            return policy, v, i + 1

    return policy, v, max_iterations


def value_iteration(env, gamma=0.99, eps=1e-10, max_iterations=100000):
    base_env = env.unwrapped
    nS = env.observation_space.n
    nA = env.action_space.n
    v = np.zeros(nS, dtype=np.float64)

    for i in range(max_iterations):
        prev_v = v.copy()

        for s in range(nS):
            q_sa = np.zeros(nA, dtype=np.float64)

            for a in range(nA):
                q_sa[a] = sum(
                    p * (r + (0.0 if terminated else gamma * prev_v[s_]))
                    for p, s_, r, terminated in base_env.P[s][a]
                )

            v[s] = np.max(q_sa)

        if np.max(np.abs(prev_v - v)) < eps:
            return v, i + 1

    return v, max_iterations


def print_policy(policy, shape=(8, 8)):
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    grid = np.array([arrows[int(a)] for a in policy]).reshape(shape)
    for row in grid:
        print(" ".join(row))


if __name__ == "__main__":
    gamma = 0.99

    # Slippery 8x8 FrozenLake
    env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True)

    # Policy Iteration
    pi_policy, pi_v, pi_iters = policy_iteration(env, gamma=gamma)
    pi_reward, pi_steps = evaluate_policy(env, pi_policy, gamma=1.0, n=5000)

    # Value Iteration
    vi_v, vi_iters = value_iteration(env, gamma=gamma)
    vi_policy = extract_policy(env, vi_v, gamma=gamma)
    vi_reward, vi_steps = evaluate_policy(env, vi_policy, gamma=1.0, n=5000)

    print("=== Policy Iteration ===")
    print("Iterations to converge:", pi_iters)
    print("Average reward:", pi_reward)
    print("Average steps :", pi_steps)
    print_policy(pi_policy)

    print("\n=== Value Iteration ===")
    print("Iterations to converge:", vi_iters)
    print("Average reward:", vi_reward)
    print("Average steps :", vi_steps)
    print_policy(vi_policy)

    print("\n=== Comparison ===")
    print("Same policy?       ", np.array_equal(pi_policy, vi_policy))
    print("Max |V_pi - V_vi| =", np.max(np.abs(pi_v - vi_v)))

=== Policy Iteration ===
Iterations to converge: 12
Average reward: 0.6254
Average steps : 72.5432
↑ → → → → → → →
↑ ↑ ↑ ↑ ↑ → → ↓
↑ ↑ ← ← → ↑ → ↓
↑ ↑ ↑ ↓ ← ← → →
← ↑ ← ← → ↓ ↑ →
← ← ← ↓ ↑ ← ← →
← ← → ← ← ← ← →
← ↓ ← ← ↓ → ↓ ←

=== Value Iteration ===
Iterations to converge: 662
Average reward: 0.6358
Average steps : 72.5296
↑ → → → → → → →
↑ ↑ ↑ ↑ ↑ → → ↓
↑ ↑ ← ← → ↑ → ↓
↑ ↑ ↑ ↓ ← ← → →
← ↑ ← ← → ↓ ↑ →
← ← ← ↓ ↑ ← ← →
← ← ↓ ← ← ← ← →
← ↓ ← ← ↓ → ↓ ←

=== Comparison ===
Same policy?        False
Max |V_pi - V_vi| = 6.351807968485446e-12


**Run the program which solves the Frozen Lake problem using the Policy Iteration and Value Iteration algorithms, respectively. Which method converges to the optimal solution in fewer steps? Check the execution time of each program using the time command. What can you conclude regarding the complexity of each algorithm?**



In [10]:
%time pi_policy, pi_V, pi_iters = policy_iteration(env, gamma=gamma)
%time vi_V, vi_iters = value_iteration(env, gamma=gamma)

CPU times: user 779 ms, sys: 1.17 ms, total: 780 ms
Wall time: 790 ms
CPU times: user 481 ms, sys: 0 ns, total: 481 ms
Wall time: 481 ms


In this example, Policy Iteration reaches convergence much faster *in terms of number of iterations*. This agrees with theory that says: Policy Iteration alternates between policy evaluation and policy improvement, and it converges to an optimal policy in finite steps for finite state and action spaces.

Value Iteration, on the other hand, updates the value function repeatedly through Bellman updates until the values become stable. In theory, this is described as successive approximations $J_n(i) \rightarrow J_{n+1}(i)$ continuing until convergence.

However, when we check the actual execution time using %time, Value Iteration was faster in this experiment: about 481 ms, compared with 790 ms for Policy Iteration.

So the conclusion is:

Policy Iteration is better in terms of number of iterations, but each iteration is computationally heavier. Value Iteration needs many more iterations, but each one is cheaper. In this Frozen Lake experiment, Value Iteration was faster overall, even though it required more iterations.

Also, both methods practically reached the same optimal value function, since:

$Max |V_{pi} - V_{vi}| = 6.35e-12$

This difference is almost zero. The policies are not exactly identical, but that is acceptable because there may be more than one optimal action in some states. Therefore, the two algorithms can produce different-looking policies while still solving the same optimization problem.